In [1]:
from datasets import load_dataset

ds = load_dataset("phonix-db/phonix-full")

/Users/liuchang/projects/foundation_model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
filtered_train = ds["train"].filter(
    lambda row: (
row["kp[W/mK]"] is not None
and row["klat[W/mK]"] is not None
and row["fc2_error[%]"] is not None
and row["fc3_error[%]"] is not None
and abs(row["fc2_error[%]"]) <= 10.0
and abs(row["fc3_error[%]"]) <= 10.0
and row["klat[W/mK]"] <= 2000
    )
)

filtered_train

Dataset({
    features: ['mp_id', 'input_dir', 'formula', 'spg_number', 'natoms_prim', 'natoms_conv', 'natoms_sc', 'trans_conv2prim', 'trans_conv2sc', 'structure', 'volume[A^3]', 'nac', 'volume_relaxation', 'scph', 'four', 'kappa_type', 'qmesh', 'kp[W/mK]', 'kc[W/mK]', 'klat[W/mK]', 'min_phfreq[cm^-1]', 'max_phfreq[cm^-1]', 'fc2_error[%]', 'fc3_error[%]', 'calc_time[sec]'],
    num_rows: 7433
})

In [3]:
filtered_train = filtered_train.to_pandas()[
    ['mp_id', 'formula', 'spg_number', 'volume[A^3]', 'kp[W/mK]', 'kc[W/mK]', 'klat[W/mK]', 'fc3_error[%]']
].drop_duplicates(subset=['mp_id'])
filtered_train.to_parquet("../phonix-db-filtered.parquet", index=False)

filtered_train

,mp_id,formula,spg_number,volume[A^3],kp[W/mK],kc[W/mK],klat[W/mK],fc3_error[%]
0,mp-1000,BaTe,225,84.639462,5.385133,0.033300,5.418433,0.210506
1,mp-147,Se,148,149.806129,1.416800,0.117300,1.534100,1.112320
3,mp-149,Si,227,40.165696,121.608400,0.026367,121.634767,0.065174
5,mp-154,N2,198,194.109334,0.025600,0.184500,0.210100,0.262661
6,mp-160,B,166,85.954429,101.501700,0.238800,101.740500,0.370470
...,...,...,...,...,...,...,...,...
7428,mp-1390618,SnO2,166,39.128256,34.342267,0.088067,34.430333,0.564014
7429,mp-1391273,ZnNiF6,148,89.155276,4.018267,0.261333,4.279600,0.616551
7430,mp-1403065,TiZnF6,148,106.884472,1.928867,0.406933,2.335800,0.703460
7431,mp-1423122,Rb2SnTe5,108,571.423863,0.369767,0.088933,0.458700,0.668410


---

Process NEMAD data

In [4]:
import pandas as pd

superconductor = pd.read_csv("../NEMAD/superconductor_materials_unique.csv")
superconductor = superconductor[['Chemical_Composition', 'Median_Tc_By_Composition_K', 'Pressure', 'Crystal_Structure', 'Space_Group', 'Experimental']]
superconductor["Experimental"] = superconductor["Experimental"].apply(
    lambda x: (
        True
        if isinstance(x, str) and x.strip().strip("'").lower() == "true"
        else (
            False
            if isinstance(x, str) and x.strip().strip("'").lower() in {"false", "theoretical", "dft calculation"}
            else None
        )
    )
)
superconductor = superconductor.dropna(subset=['Chemical_Composition', 'Median_Tc_By_Composition_K', "Experimental"])

superconductor = superconductor.rename(columns={
    "Chemical_Composition": "composition",
    "Median_Tc_By_Composition_K": "Tc[K]",
    "Pressure": "pressure[GPa]",
    "Crystal_Structure": "crystal_structure",
    "Space_Group": "space_group",
    "Experimental": "experimental"
})



superconductor.to_parquet("../superconductor_nemad.parquet", index=False)
superconductor

,composition,Tc[K],pressure[GPa],crystal_structure,space_group,experimental
0,AcBH16,39.00,100 GPa,Orthorhombic,Pm21b,False
1,AcBH7,82.50,200 GPa,Trigonal,P-3m1,False
2,AcBH8,44.50,200 GPa,Monoclinic,P21/m,False
4,AcB2H14,42.00,200 GPa,Orthorhombic,Pnm21,False
5,AcBeH10,165.00,300 GPa,Cubic,Fm-3m,False
...,...,...,...,...,...,...
19051,V67Zr33,8.00,NaN,Cubic Laves phase,NaN,True
19052,Zr64V36,1.30,NaN,NaN,NaN,True
19053,ZrW2,1.16,NaN,Laves phase,NaN,True
19054,ZnY,0.30,NaN,Cubic,NaN,True


In [5]:
magnetic = pd.read_csv("../NEMAD/magnetic_materials.csv")
magnetic = magnetic[['New_Column_Concatenated', 'Magnetic_Moment', 'Magnetization', 'Curie', 'Curie(Tc)', 'Neel', 'Neel(Tn)','Space_Group', 'Experimental']]
magnetic = magnetic.dropna(subset=['New_Column_Concatenated']).drop_duplicates(subset=['New_Column_Concatenated'])

magnetic

/var/folders/ms/trlx2qj16hg7p3qd0k66yc340000gn/T/ipykernel_56549/1290270689.py:1: DtypeWarning: Columns (10,15,16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  magnetic = pd.read_csv("../NEMAD/magnetic_materials.csv")


,New_Column_Concatenated,Magnetic_Moment,Magnetization,Curie,Curie(Tc),Neel,Neel(Tn),Space_Group,Experimental
0,O3.0Mn1Sr0.5Nd0.5,150μB,NaN,NaN,NaN,50,50K,NaN,True
1,O4.0Fe3.0,NaN,32-58 emu/g,858.15,585 °C,NaN,NaN,NaN,True
2,Si11.0Gd3.0Pt23.0,7.00±0.10 μB/R,NaN,42,42±1 K,NaN,NaN,Fm3̄m,True
3,Si11.0Tb3.0Pt23.0,5.86±0.10 μB/R,NaN,25,25±1 K,NaN,NaN,Fm3̄m,True
4,Si11.0Dy3.0Pt23.0,7.08±0.10 μB/R,NaN,15.99,15.99±0.10 K,NaN,NaN,Fm3̄m,True
...,...,...,...,...,...,...,...,...,...
26689,O6.0Co1Sr2.0Os1,NaN,NaN,NaN,NaN,89,"108 K, 70 K",NaN,False
26691,O6.0Ca2.0Co1Os1,NaN,NaN,145,145 K,NaN,NaN,NaN,False
26692,O6.0Ca2.0Ni1Os1,NaN,NaN,175,175 K,NaN,NaN,NaN,False
26700,Mn34.0Ni50.0Ga4.0In12.0,NaN,Relatively weak saturation magnetization under...,295,295 K,NaN,NaN,NaN,True
